In [1]:
import sys
from absl import flags
from ml_collections.config_flags import config_flags

sys.argv = [
    "",
    "--config=td_sa_stack/config.py",
]

config_flags.DEFINE_config_file("config", None, "Training configuration.", lock_config=True)
# flags.DEFINE_string("workdir", None, "Work directory.")
# flags.DEFINE_enum("mode", None, ["train", "eval", "fid_stats"], "Running mode: train, eval or fid_stats")
# flags.DEFINE_string("eval_folder", "eval", "The folder name for storing evaluation results")


FLAGS = flags.FLAGS
FLAGS(sys.argv)

config = FLAGS.config

In [2]:
config.model

activation: swish
name: RegressionInceptionNetV1
optimizer: adamw
optimizer_weight_decay: 1.0e-05

In [3]:
import tensorflow as tf
tf.config.experimental.set_visible_devices([], "GPU") # Отключение GPU для TensorFlow

import os
import jax
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.8'

from td_sa_stack import get_dataset, TrainerModule, RegressionInceptionNetV1

%load_ext autoreload
%autoreload 2

2025-06-15 14:28:38.641174: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-15 14:28:38.662971: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749997718.686529    7222 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749997718.694357    7222 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749997718.716656    7222 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [4]:
jax.devices()

[CudaDevice(id=0), CudaDevice(id=1)]

In [ ]:
train_ds, _, _ = get_dataset(config, uniform_dequantization=config.data.uniform_dequantization)

trainer = TrainerModule(config=config,
                        model_class=RegressionInceptionNetV1,
                        version=10)

Batch dimensions: [2, 512]
Initializing model with batch shape: (512, 32, 32, 6)


2025-06-15 14:29:04.139117: E external/xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv.1 = (f32[512,32,32,192]{3,2,1,0}, u8[0]{0}) custom-call(%Arg_0.1, %bitcast.7), window={size=3x3 pad=1_1x1_1}, dim_labels=b01f_o01i->b01f, custom_call_target="__cudnn$convForward", metadata={op_name="jit(conv_general_dilated)/jit(main)/conv_general_dilated" source_file="/workspace/.venv/lib/python3.12/site-packages/flax/linen/linear.py" source_line=694}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2025-06-15 14:29:04.515828: E external/xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.376951695s
Trying algorithm eng0{} for conv %cudnn-conv.1 = (f32[512,32,32,192]{3,2,1,0}, u8[0]{0}) custom-call(%Arg_0.1, %bitcast.7), window={si

In [ ]:
trainer.train_model(train_ds=train_ds)

2025-06-15 14:11:42.062069: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2025-06-15 14:12:04.096838: E external/xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bw-input.35 = (f32[512,32,32,128]{3,2,1,0}, u8[0]{0}) custom-call(%add.12988, %bitcast.1992), window={size=3x3 pad=1_1x1_1}, dim_labels=b01f_o01i->b01f, custom_call_target="__cudnn$convBackwardInput", metadata={op_name="pmap(train_step_pmap)/jit(main)/transpose(jvp(RegressionInceptionNetV1))/InceptionBlock_1/conv_b_3x3_2/ConvNxN/conv_general_dilated" source_file="/workspace/.venv/lib/python3.12/site-packages/flax/linen/linear.py" source_line=694}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reifica